# USOPC Portfolio Investment Analytics — LA 2028 Summer Games

> **Public methodology repository.** Athlete archetypes are composites across disciplines and cycles — not individuals.
> Framework formalizes the portfolio investment approach described by Rocky Harris, USOPC Chief of Sport & Athlete Services.
> **Summer Games focus:** Gymnastics, Swimming, Track & Field, Soccer, Basketball, Diving, Rowing.
> Private repository contains named athlete data and full validation results.

**Archetype key:**
- `DOMINANT_FIRST` — dominant pre-Games, first Olympics, highest preparation gap signal
- `DOMINANT_EXP` — dominant pre-Games, experienced, preparation complete
- `ELITE_RETURN` — returning after adversity, high variance, support-dependent
- `ASCENDING` — program on upward trajectory, development investment thesis
- `MAINTENANCE` — perennially dominant program, protection investment
- `VOLATILE` — high variance athlete, wide outcome distribution
- `DEVELOPMENT` — emerging athlete, low floor, high ceiling

**LA 2028 context:** Home Games on home soil. Maximum commercial and competitive stakes.
The Harris portfolio framework is most consequential when applied to the largest Summer cycle in a generation.


# 🏅 USOPC Analytics Pipeline
## Monte Carlo · Performance Prediction · LP/IP Optimization · Policy Simulation · Validation

**One notebook. Six sections. One athletes_df that grows through each stage.**

| Section | What it does | Key output |
|---------|-------------|------------|
| 0 — Ingestion | Scrape ISU, FIS, World Athletics pre-Olympic results | Raw score DataFrames cached to disk |
| 1 — Features | Engineer age, career arc, consistency, NLP load | `athletes_df` with all features |
| 2 — Prediction | Gradient boosting + quantile regression | `pred_mean`, `pred_std` columns added |
| 3 — Monte Carlo | Simulate P(gold) using predicted distributions | `p_gold`, `p_medal` columns added |
| 4 — Optimization | LP/IP portfolio, efficient frontier | `selected` column + frontier DataFrame |
| 5 — Policy Sim | Investment → performance feedback loop | Counterfactual scenarios |
| 6 — Validation | Brier, MAE, skill score vs naive baseline | Error dashboard |

**Data flow:** Everything stays in memory as one DataFrame.  
**Only disk writes:** Raw scraped data (cached), final charts, master results CSV.


## Setup

In [ ]:
!pip install requests beautifulsoup4 lxml pandas numpy scipy matplotlib seaborn scikit-learn lightgbm textblob pulp -q

import os, time, json, warnings
import requests
from bs4 import BeautifulSoup

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from scipy.stats import norm
from scipy.optimize import minimize
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import QuantileRegressor
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, brier_score_loss
import lightgbm as lgb
from textblob import TextBlob
import pulp

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

# ── Paths ─────────────────────────────────────────────────────
RAW_DIR    = 'data/raw'
OUTPUT_DIR = 'data/outputs'
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Constants ─────────────────────────────────────────────────
N_SIMS        = 10_000
RATE_LIMIT_S  = 1.5   # seconds between scrape requests
SESSION       = requests.Session()
SESSION.headers.update({
    'User-Agent': 'USOPC-Research/1.0 (academic portfolio project)'
})

print('✅ Setup complete')
print(f'   Raw data cache: {RAW_DIR}/')
print(f'   Outputs:        {OUTPUT_DIR}/')


---
## Section 0 — Data Ingestion

Scrapes pre-Olympic season results from public sports federation websites.  
**Caches to disk** — checks for existing file before scraping. Run once per season.

Sources:
- **ISU** — results.isu.org — Figure Skating, Speed Skating
- **FIS** — fis-ski.com — Alpine, Freestyle, Snowboard, Cross-Country
- **World Athletics** — worldathletics.org — Track & Field season top lists
- **World Aquatics** — worldaquatics.com — Swimming
- **Wikipedia API** — athlete biographical features (age, career start, Olympic history)


In [ ]:
# ── Scraper utilities ─────────────────────────────────────────

def cached_get(url, cache_path, force_refresh=False):
    """
    GET url, cache response HTML to disk.
    Returns BeautifulSoup object.
    Respects rate limit between live requests.
    """
    if os.path.exists(cache_path) and not force_refresh:
        with open(cache_path, 'r', encoding='utf-8') as f:
            html = f.read()
        print(f'  [cache] {os.path.basename(cache_path)}')
    else:
        print(f'  [fetch] {url[:80]}...')
        try:
            resp = SESSION.get(url, timeout=15)
            resp.raise_for_status()
            html = resp.text
            with open(cache_path, 'w', encoding='utf-8') as f:
                f.write(html)
            time.sleep(RATE_LIMIT_S)
        except Exception as e:
            print(f'  [error] {e} — returning empty')
            return None
    return BeautifulSoup(html, 'lxml')


def parse_table(soup, table_index=0):
    """Parse first HTML table to DataFrame."""
    if soup is None:
        return pd.DataFrame()
    tables = soup.find_all('table')
    if not tables or table_index >= len(tables):
        return pd.DataFrame()
    try:
        return pd.read_html(str(tables[table_index]))[0]
    except Exception as e:
        print(f'  [parse error] {e}')
        return pd.DataFrame()


print('✅ Scraper utilities ready')
print()
print('Cache policy: reads from disk if file exists, fetches live if not')
print('Rate limit:   1.5 seconds between requests')


In [ ]:
# ── Wikipedia API — athlete biographical features ─────────────

def get_athlete_wiki(athlete_name):
    """
    Pull biographical data from Wikipedia API.
    Returns dict with birth_year, nationality, career_start, olympic_history.
    """
    cache_path = f'{RAW_DIR}/wiki_{athlete_name.replace(" ","_")}.json'

    if os.path.exists(cache_path):
        with open(cache_path) as f:
            return json.load(f)

    url = 'https://en.wikipedia.org/api/rest_v1/page/summary/' +           athlete_name.replace(' ', '_')
    try:
        resp = SESSION.get(url, timeout=10)
        resp.raise_for_status()
        data = resp.json()
        result = {
            'name':        athlete_name,
            'description': data.get('description', ''),
            'extract':     data.get('extract', '')[:500],
            'wiki_url':    data.get('content_urls', {}).get('desktop', {}).get('page', '')
        }
        with open(cache_path, 'w') as f:
            json.dump(result, f)
        time.sleep(RATE_LIMIT_S)
        return result
    except Exception as e:
        print(f'  [wiki error] {athlete_name}: {e}')
        return {'name': athlete_name, 'description': '', 'extract': '', 'wiki_url': ''}


# Test Wikipedia API
print('Testing Wikipedia API...')
malinin_wiki = get_athlete_wiki('DOMINANT_FIRST')
print(f'  DOMINANT_FIRST: {malinin_wiki["description"]}')
print(f'  Extract: {malinin_wiki["extract"][:200]}...')


In [ ]:
# ── ISU Figure Skating — season results scraper ───────────────

def scrape_isu_season(season='2025-26', discipline='men', force_refresh=False):
    """
    Scrape ISU season results for figure skating.
    season: '2025-26', '2024-25', etc.
    discipline: 'men', 'women', 'pairs', 'dance'
    Returns DataFrame with columns: rank, name, nation, score, competition, date
    """
    # ISU results search URL structure
    season_code = season.replace('-', '')[:6]
    cache_path = f'{RAW_DIR}/isu_{discipline}_{season_code}.html'

    # ISU season standings page
    url = (f'https://results.isu.org/results/season{season_code}/'
           f'results/cat000{"MOE" if discipline=="men" else "WOE"}.htm')

    print(f'ISU {discipline} {season}:')
    soup = cached_get(url, cache_path, force_refresh)

    if soup is None:
        print('  Returning placeholder — populate manually from results.isu.org')
        return _isu_placeholder(discipline, season)

    # Parse results table
    df = parse_table(soup)
    if df.empty:
        print('  Parse failed — returning placeholder')
        return _isu_placeholder(discipline, season)

    df['discipline'] = f'Figure Skating {discipline.title()}'
    df['season']     = season
    df['source']     = 'ISU'
    return df


def _isu_placeholder(discipline, season):
    """
    Placeholder data structure matching what live scrape would return.
    REPLACE with real data from results.isu.org
    Season best scores for key athletes — pre-Olympic 2025-26 season.
    """
    if discipline == 'men':
        return pd.DataFrame([
            # 2025-26 pre-Games_2026 season results
            # Source: results.isu.org — replace PLACEHOLDERs with actuals
            dict(name='DOMINANT_FIRST',        nation='USA', competition='Grand Prix Final',
                 score=317.45, date='2025-12-05', season=season,
                 discipline='Figure Skating Men', source='ISU_PLACEHOLDER'),
            dict(name='DOMINANT_FIRST',        nation='USA', competition='Skate America',
                 score=309.82, date='2025-10-18', season=season,
                 discipline='Figure Skating Men', source='ISU_PLACEHOLDER'),
            dict(name='DOMINANT_FIRST',        nation='USA', competition='NHK Trophy',
                 score=312.17, date='2025-11-08', season=season,
                 discipline='Figure Skating Men', source='ISU_PLACEHOLDER'),
            dict(name='DOMINANT_FIRST',        nation='USA', competition='US Championships',
                 score=321.03, date='2026-01-22', season=season,
                 discipline='Figure Skating Men', source='ISU_PLACEHOLDER'),
            dict(name='VOLATILE',   nation='KAZ', competition='Grand Prix Final',
                 score=289.41, date='2025-12-05', season=season,
                 discipline='Figure Skating Men', source='ISU_PLACEHOLDER'),
            dict(name='DOMINANT_EXP',       nation='JPN', competition='NHK Trophy',
                 score=295.62, date='2025-11-08', season=season,
                 discipline='Figure Skating Men', source='ISU_PLACEHOLDER'),
            dict(name='DEVELOPMENT',           nation='JPN', competition='Grand Prix Final',
                 score=287.33, date='2025-12-05', season=season,
                 discipline='Figure Skating Men', source='ISU_PLACEHOLDER'),
        ])
    elif discipline == 'women':
        return pd.DataFrame([
            dict(name='ASCENDING',           nation='USA', competition='US Championships',
                 score=218.45, date='2026-01-22', season=season,
                 discipline='Figure Skating Women', source='ISU_PLACEHOLDER'),
            dict(name='DEVELOPMENT',      nation='USA', competition='Grand Prix Final',
                 score=221.33, date='2025-12-05', season=season,
                 discipline='Figure Skating Women', source='ISU_PLACEHOLDER'),
            dict(name='DOMINANT_EXP',      nation='JPN', competition='NHK Trophy',
                 score=229.18, date='2025-11-08', season=season,
                 discipline='Figure Skating Women', source='ISU_PLACEHOLDER'),
        ])
    return pd.DataFrame()


# Run ISU scrapers
isu_men   = scrape_isu_season('2025-26', 'men')
isu_women = scrape_isu_season('2025-26', 'women')

print()
print(f'ISU Men results: {len(isu_men)} rows')
print(f'ISU Women results: {len(isu_women)} rows')
print()
print(isu_men[['name','competition','score','date']].to_string(index=False))


In [ ]:
# ── FIS Alpine Skiing — World Cup season results ─────────────

def scrape_fis_alpine(season='2025-26', event='SL', gender='L', force_refresh=False):
    """
    Scrape FIS Alpine World Cup results.
    event: SL=Slalom, GS=Giant Slalom, DH=Downhill, SG=Super-G, AC=Alpine Combined
    gender: L=Ladies, M=Men
    Returns DataFrame with race results for the season.
    """
    season_id = season.replace('-','')
    cache_path = f'{RAW_DIR}/fis_alpine_{gender}_{event}_{season_id}.html'

    # FIS results page
    url = (f'https://www.fis-ski.com/DB/alpine-skiing/cup-standings.html'
           f'?sectorcode=AL&seasoncode={season_id}&categorycode=WC'
           f'&disciplinecode={event}&gendercode={gender}&stylescode=&nationcode='
           f'&seasonmonth=X-{season_id}&saveselection=-1&seasonselection=')

    print(f'FIS Alpine {gender} {event} {season}:')
    soup = cached_get(url, cache_path, force_refresh)

    if soup is None or parse_table(soup).empty:
        print('  Returning placeholder — populate from fis-ski.com')
        return _fis_placeholder(season, event, gender)

    df = parse_table(soup)
    df['discipline'] = f'Alpine {"Women" if gender=="L" else "Men"} {event}'
    df['season']     = season
    df['source']     = 'FIS'
    return df


def _fis_placeholder(season, event, gender):
    """Placeholder FIS results — replace with actuals from fis-ski.com"""
    if gender == 'L' and event == 'SL':
        return pd.DataFrame([
            dict(name='DOMINANT_EXP', nation='USA', rank=1,
                 points=600, wins=4, season=season,
                 discipline='Alpine Women SL', source='FIS_PLACEHOLDER'),
            dict(name='Petra Vlhova',     nation='SVK', rank=2,
                 points=512, wins=2, season=season,
                 discipline='Alpine Women SL', source='FIS_PLACEHOLDER'),
            dict(name='Katharina Liensberger', nation='AUT', rank=3,
                 points=441, wins=1, season=season,
                 discipline='Alpine Women SL', source='FIS_PLACEHOLDER'),
        ])
    elif gender == 'M' and event == 'DH':
        return pd.DataFrame([
            dict(name='Marco Odermatt',   nation='SUI', rank=1,
                 points=720, wins=5, season=season,
                 discipline='Alpine Men DH', source='FIS_PLACEHOLDER'),
            dict(name='Cyprien Sarrazin', nation='FRA', rank=2,
                 points=580, wins=2, season=season,
                 discipline='Alpine Men DH', source='FIS_PLACEHOLDER'),
        ])
    return pd.DataFrame()


fis_women_sl = scrape_fis_alpine('2025-26', 'SL', 'L')
fis_men_dh   = scrape_fis_alpine('2025-26', 'DH', 'M')

print()
print(f'FIS Women SL: {len(fis_women_sl)} rows')
print(f'FIS Men DH:   {len(fis_men_dh)} rows')


In [ ]:
# ── Speed Skating — ISU speed skating results ────────────────

def scrape_speed_skating(season='2025-26', event='500m', gender='M', force_refresh=False):
    """
    Scrape ISU Speed Skating season results.
    Returns DataFrame with times and rankings.
    """
    cache_path = f'{RAW_DIR}/speedskating_{gender}_{event}_{season.replace("-","")}.html'

    url = (f'https://speedskatingresults.com/index.php?cat=9'
           f'&season={season}&distance={event}&gender={gender}')

    print(f'Speed Skating {gender} {event} {season}:')
    soup = cached_get(url, cache_path, force_refresh)

    if soup is None or parse_table(soup).empty:
        print('  Returning placeholder — populate from speedskatingresults.com')
        return _speed_placeholder(season, event, gender)

    df = parse_table(soup)
    df['discipline'] = f'Speed Skating {gender} {event}'
    df['season']     = season
    df['source']     = 'ISU_Speed'
    return df


def _speed_placeholder(season, event, gender):
    if gender == 'M' and event == '500m':
        return pd.DataFrame([
            dict(name='MAINTENANCE',      nation='USA', rank=1,
                 time=33.61, points=None, season=season,
                 discipline='Speed Skating M 500m', source='ISU_PLACEHOLDER'),
            dict(name='Laurent Dubreuil',  nation='CAN', rank=2,
                 time=33.74, points=None, season=season,
                 discipline='Speed Skating M 500m', source='ISU_PLACEHOLDER'),
            dict(name='Tatsuya Shinhama',  nation='JPN', rank=3,
                 time=33.82, points=None, season=season,
                 discipline='Speed Skating M 500m', source='ISU_PLACEHOLDER'),
        ])
    return pd.DataFrame()


speed_m_500 = scrape_speed_skating('2025-26', '500m', 'M')
print(f'Speed Skating M 500m: {len(speed_m_500)} rows')


In [ ]:
# ── World Athletics — seasonal top lists ─────────────────────

def scrape_world_athletics(event='100m', gender='men', year=2025, force_refresh=False):
    """
    Scrape World Athletics seasonal top list.
    Returns DataFrame with athletes ranked by season best.
    """
    cache_path = f'{RAW_DIR}/worldathletics_{gender}_{event}_{year}.html'

    url = (f'https://worldathletics.org/records/toplists/sprints/{event}/outdoor'
           f'/{gender}/senior/{year}?regionType=world&page=1&bestResultsOnly=true')

    print(f'World Athletics {gender} {event} {year}:')
    soup = cached_get(url, cache_path, force_refresh)

    if soup is None or parse_table(soup).empty:
        print('  Returning placeholder — populate from worldathletics.org')
        return _athletics_placeholder(event, gender, year)

    df = parse_table(soup)
    df['discipline'] = f'Athletics {gender.title()} {event}'
    df['season']     = f'{year}-{str(year+1)[2:]}'
    df['source']     = 'WorldAthletics'
    return df


def _athletics_placeholder(event, gender, year):
    if event == '100m' and gender == 'men':
        return pd.DataFrame([
            dict(name='DOMINANT_EXP',        nation='USA', rank=1,
                 mark='9.83', wind='+0.8', season=f'{year}-{str(year+1)[2:]}',
                 discipline='Athletics Men 100m', source='WA_PLACEHOLDER'),
            dict(name='VOLATILE',       nation='USA', rank=2,
                 mark='9.85', wind='+1.1', season=f'{year}-{str(year+1)[2:]}',
                 discipline='Athletics Men 100m', source='WA_PLACEHOLDER'),
            dict(name='Kishane Thompson',  nation='JAM', rank=3,
                 mark='9.87', wind='+0.5', season=f'{year}-{str(year+1)[2:]}',
                 discipline='Athletics Men 100m', source='WA_PLACEHOLDER'),
        ])
    return pd.DataFrame()


athletics_100m = scrape_world_athletics('100m', 'men', 2025)
print(f'Athletics Men 100m: {len(athletics_100m)} rows')


In [ ]:
# ── Olympedia — historical Olympic results (backtesting ground truth) ──

def scrape_olympedia_event(games_slug, event_slug, force_refresh=False):
    """
    Scrape Olympedia for actual Olympic results.
    Used in Section 6 (Validation) as ground truth.
    games_slug: e.g. '2022_W' for Games_2022 Winter
    event_slug: e.g. 'Figure_Skating_Men' 
    """
    cache_path = f'{RAW_DIR}/olympedia_{games_slug}_{event_slug}.html'
    url = f'https://www.olympedia.org/results/{games_slug}/{event_slug}'

    print(f'Olympedia {games_slug} {event_slug}:')
    soup = cached_get(url, cache_path, force_refresh)

    if soup is None or parse_table(soup).empty:
        print('  Returning placeholder — populate from olympedia.org')
        return _olympedia_placeholder(games_slug, event_slug)

    df = parse_table(soup)
    df['games'] = games_slug
    df['event'] = event_slug
    df['source'] = 'Olympedia'
    return df


def _olympedia_placeholder(games_slug, event_slug):
    """Known Olympic results — ground truth for validation."""
    results = {
        ('2022_W', 'Figure_Skating_Men'): [
            dict(rank=1, name='DOMINANT_EXP',    nation='USA', score=332.60,
                 games='Games_2022', event='Figure Skating Men'),
            dict(rank=2, name='DOMINANT_EXP',  nation='JPN', score=310.05,
                 games='Games_2022', event='Figure Skating Men'),
            dict(rank=3, name='Shoma Uno',      nation='JPN', score=293.00,
                 games='Games_2022', event='Figure Skating Men'),
        ],
        ('2026_W', 'Figure_Skating_Men'): [
            dict(rank=1, name='VOLATILE', nation='KAZ', score=289.41,
                 games='Games_2026', event='Figure Skating Men'),
            dict(rank=2, name='DOMINANT_EXP',    nation='JPN', score=278.33,
                 games='Games_2026', event='Figure Skating Men'),
            dict(rank=3, name='DEVELOPMENT',        nation='JPN', score=275.18,
                 games='Games_2026', event='Figure Skating Men'),
            dict(rank=8, name='DOMINANT_FIRST',     nation='USA', score=234.82,
                 games='Games_2026', event='Figure Skating Men'),
        ],
    }
    key = (games_slug, event_slug)
    rows = results.get(key, [])
    return pd.DataFrame(rows) if rows else pd.DataFrame()


# Scrape known Olympic results for validation
beijing_skating   = scrape_olympedia_event('2022_W', 'Figure_Skating_Men')
milan_skating     = scrape_olympedia_event('2026_W', 'Figure_Skating_Men')

print()
print('Games_2026 Men Figure Skating results:')
print(milan_skating[['rank','name','nation','score']].to_string(index=False))


In [ ]:
# ── Consolidate raw data into normalized scores ───────────────

def normalize_scores(df, discipline):
    """
    Convert raw results to normalized score (0-100) within discipline.
    Different sports have incompatible scoring systems — normalization
    allows cross-sport comparison in the prediction model.
    
    Figure skating: raw total score / max possible * 100
    Alpine skiing: inverse of time-based points (higher rank = higher score)
    Speed skating: inverse of time (faster = higher score)
    Athletics: inverse of time/distance percentile
    """
    df = df.copy()

    if 'Figure Skating' in discipline:
        max_score = 400.0  # approximate max total score
        df['norm_score'] = (df['score'] / max_score * 100).clip(0, 100)

    elif 'Alpine' in discipline:
        if 'points' in df.columns:
            max_pts = df['points'].max() + 1e-9
            df['norm_score'] = (df['points'] / max_pts * 100).clip(0, 100)
        else:
            df['norm_score'] = 50.0  # placeholder

    elif 'Speed Skating' in discipline:
        if 'time' in df.columns:
            # Faster time = higher score
            min_time = df['time'].min()
            max_time = df['time'].max()
            df['norm_score'] = ((max_time - df['time']) /
                                (max_time - min_time + 1e-9) * 100).clip(0, 100)
        else:
            df['norm_score'] = 50.0

    elif 'Athletics' in discipline:
        # For sprint events: faster = higher score
        df['norm_score'] = 50.0  # placeholder — parse mark string properly in production

    else:
        df['norm_score'] = 50.0

    return df


# Normalize all raw data
isu_men_norm   = normalize_scores(isu_men, 'Figure Skating Men')
isu_women_norm = normalize_scores(isu_women, 'Figure Skating Women')
speed_norm     = normalize_scores(speed_m_500, 'Speed Skating M 500m')

print('✅ Raw data normalized')
print()
print('ISU Men normalized scores:')
if 'norm_score' in isu_men_norm.columns:
    print(isu_men_norm[['name','competition','score','norm_score']].to_string(index=False))


---
## Section 1 — Feature Engineering

Build `athletes_df` — one row per athlete per Games cycle.  
This is the master DataFrame that grows through every section.


In [ ]:
# ── Build per-athlete season summary ──────────────────────────

def build_athlete_season_summary(raw_dfs, games, season):
    """
    From raw results DataFrames, build one row per athlete with:
    - mean_pre, std_pre, best_pre: season score statistics
    - comps_pre: number of competitions
    - days_since_last: days between last competition and Games opening
    - win_streak: consecutive wins entering Games
    """
    all_results = pd.concat([df for df in raw_dfs if not df.empty], ignore_index=True)

    if all_results.empty or 'norm_score' not in all_results.columns:
        return pd.DataFrame()

    records = []
    for name, group in all_results.groupby('name'):
        nation = group['nation'].iloc[0] if 'nation' in group.columns else 'UNK'
        discipline = group['discipline'].iloc[0] if 'discipline' in group.columns else 'Unknown'

        scores = group['norm_score'].dropna().values
        if len(scores) == 0:
            continue

        # Sort by date if available
        if 'date' in group.columns:
            group = group.sort_values('date')
            last_date = pd.to_datetime(group['date'].iloc[-1], errors='coerce')
        else:
            last_date = None

        records.append({
            'athlete_id':   f'{name.lower().replace(" ","_")}_{discipline.lower().replace(" ","_")}_{games.replace(" ","_").lower()}',
            'name':         name,
            'nation':       nation,
            'discipline':   discipline,
            'games':        games,
            'season':       season,
            'mean_pre':     round(float(np.mean(scores)), 2),
            'std_pre':      round(float(np.std(scores)), 2) if len(scores) > 1 else 5.0,
            'best_pre':     round(float(np.max(scores)), 2),
            'comps_pre':    len(scores),
            'last_comp_date': last_date,
        })

    return pd.DataFrame(records)


# Build 2026 Games_2026 season summary
raw_2026 = [isu_men_norm, isu_women_norm, speed_norm]
season_summary_2026 = build_athlete_season_summary(raw_2026, 'Games_2026', '2025-26')

print(f'Season summary built: {len(season_summary_2026)} athletes')
print()
if not season_summary_2026.empty:
    print(season_summary_2026[['name','discipline','mean_pre','std_pre','best_pre','comps_pre']].to_string(index=False))


In [ ]:
# ── Add biographical features from Wikipedia + manual ─────────

# Manually curated biographical data
# In production: parse Wikipedia extracts for birth year, career start
# Anonymized athlete archetypes — replace with real athlete data in private repo
# Archetypes represent realistic performance profiles across disciplines
bio_data = {
    'DOMINANT_FIRST':  dict(birth_year=2004, peak_age=24, career_start=2021, prior_olympics=0),
    'ASCENDING':  dict(birth_year=2005, peak_age=22, career_start=2019, prior_olympics=0),
    'DEVELOPMENT':  dict(birth_year=2006, peak_age=21, career_start=2022, prior_olympics=0),
    'DOMINANT_EXP':  dict(birth_year=2000, peak_age=22, career_start=2017, prior_olympics=1),
    'VOLATILE':  dict(birth_year=2002, peak_age=23, career_start=2019, prior_olympics=0),
    'DOMINANT_EXP':  dict(birth_year=2002, peak_age=23, career_start=2018, prior_olympics=1),
    'DEVELOPMENT':  dict(birth_year=2001, peak_age=23, career_start=2018, prior_olympics=0),
    'DOMINANT_EXP':  dict(birth_year=1995, peak_age=26, career_start=2011, prior_olympics=3),
    'MAINTENANCE':  dict(birth_year=2004, peak_age=24, career_start=2022, prior_olympics=0),
    'DOMINANT_EXP':  dict(birth_year=1999, peak_age=23, career_start=2015, prior_olympics=1),
    'DOMINANT_EXP':  dict(birth_year=1997, peak_age=27, career_start=2016, prior_olympics=1),
    'VOLATILE':  dict(birth_year=1994, peak_age=30, career_start=2017, prior_olympics=1),
}
# Key: FS=Figure Skating, AL=Alpine, SS=Speed Skating, AT=Athletics
#      M=Men, W=Women, A/B/C/D/E=athlete index within discipline

# Win streaks entering Games (anonymized)
win_streaks = {
    'DOMINANT_FIRST':  14,
    'DOMINANT_EXP':  1,
    'MAINTENANCE':  8,
    'DOMINANT_EXP':  3,
    'DOMINANT_EXP':  5,
}

def add_bio_features(df, bio_dict, streak_dict, games_year):
    """Add biographical and win streak features to season summary."""
    df = df.copy()
    rows = []
    for _, row in df.iterrows():
        bio = bio_dict.get(row['name'], {})
        age = games_year - bio.get('birth_year', games_year - 25)
        peak = bio.get('peak_age', 24)
        rows.append({
            **row.to_dict(),
            'age':             age,
            'peak_age':        peak,
            'age_vs_peak':     age - peak,
            'seasons_elite':   games_year - bio.get('career_start', games_year - 4),
            'prior_olympics':  bio.get('prior_olympics', 0),
            'first_olympics':  int(bio.get('prior_olympics', 0) == 0),
            'win_streak':      streak_dict.get(row['name'], 0),
        })
    return pd.DataFrame(rows)


athletes_df = add_bio_features(season_summary_2026, bio_data, win_streaks, 2026)

print(f'athletes_df: {len(athletes_df)} rows × {len(athletes_df.columns)} columns')
print()
print(athletes_df[['name','age','age_vs_peak','first_olympics',
                    'win_streak','prior_olympics','mean_pre','std_pre']].to_string(index=False))


In [ ]:
# ── Engineer derived features ─────────────────────────────────

def engineer_features(df):
    df = df.copy()

    # Consistency: lower CV = more consistent in normal competition
    df['cv_pre'] = df['std_pre'] / (df['mean_pre'] + 1e-9)

    # How often do they hit their best?
    df['peak_proximity'] = df['best_pre'] / (df['mean_pre'] + 1e-9)

    # Olympic experience non-linear — first Games hardest
    df['olympic_exp_sq']  = df['prior_olympics'] ** 2

    # Recency — placeholder, update with actual days_since_last
    df['days_since_last'] = df.get('days_since_last', 14)
    df['recency_decay']   = np.exp(-df['days_since_last'] / 30)

    # Streak pressure — log to dampen extreme streaks
    df['consistency_load'] = np.log1p(df['win_streak'])

    # Career phase signal
    df['ascending']  = (df['age_vs_peak'] < 0).astype(int)
    df['elite_log']  = np.log1p(df['seasons_elite'])

    return df


athletes_df = engineer_features(athletes_df)

print('✅ Features engineered')
print()
print('Derived features for DOMINANT_FIRST:')
mal = athletes_df[athletes_df['name'] == 'DOMINANT_FIRST']
if not mal.empty:
    mal = mal.iloc[0]
    print(f'  cv_pre:          {mal.cv_pre:.3f}')
    print(f'  peak_proximity:  {mal.peak_proximity:.3f}')
    print(f'  consistency_load: {mal.consistency_load:.3f}')
    print(f'  first_olympics:  {mal.first_olympics}')
    print(f'  age_vs_peak:     {mal.age_vs_peak:+.0f}')


---
## Section 2 — Performance Prediction

**Gradient boosting** predicts expected Olympic score from pre-Games features.  
**Quantile regression** gives uncertainty intervals (p10–p90).  
**NLP sentiment** adjusts predicted std for preparation readiness.

Training data: historical Olympic cycles where we have both pre-Games season data  
AND actual Olympic results. Grows with each Games cycle.


In [ ]:
# ── Historical training data — COMPOSITE ARCHETYPES ───────────
#
# Rows represent PATTERNS not individuals.
# Each archetype blends multiple athletes across multiple Games cycles
# who share the same structural profile.
# This makes the model a pattern-recognition system, not a person-prediction system.
#
# Archetype naming convention:
#   DOMINANT_FIRST   — dominant pre-Games record, no prior Olympics, high preparation gap
#   DOMINANT_EXP     — dominant pre-Games record, experienced, preparation complete
#   ELITE_RETURN     — elite athlete returning after adversity or withdrawal
#   ELITE_PEAK       — athlete at or past physical peak, deep experience
#   ASCENDING        — program on upward trajectory, investment thesis forming
#   MAINTENANCE      — program sustaining dominance, protection investment
#   VOLATILE         — high variance athlete, wide outcome distribution
#   DEVELOPMENT      — emerging athlete, low floor, high ceiling
#
# Private repo contains named athlete data mapped to these archetypes.

training_data = pd.DataFrame([

    # ── DOMINANT_FIRST ────────────────────────────────────────
    # Pattern: top pre-Games scores, long win streak, zero Olympic experience
    # Historical instances: multiple disciplines, multiple cycles
    # Characteristic outcome: wide result distribution, underperformance more likely
    # Preparation gap is the key structural variable
    dict(name='DOMINANT_FIRST', games='Tokyo_2021', discipline='Gymnastics AA',
         mean_pre=83.5, std_pre=4.8, best_pre=88.2, comps_pre=5,
         age=19, peak_age=23, age_vs_peak=-4, seasons_elite=2,
         prior_olympics=0, first_olympics=1, win_streak=4,
         cv_pre=0.057, peak_proximity=1.058, consistency_load=0.8,
         olympic_exp_sq=0, recency_decay=0.54, ascending=1, elite_log=1.15,
         sentiment=0.42, actual_score=63.8),

    # Second instance — higher dominance, longer streak, lower sentiment
    # Preparation gap more pronounced
    dict(name='DOMINANT_FIRST', games='Paris_2024', discipline='Gymnastics AA',
         mean_pre=96.5, std_pre=3.1, best_pre=100.4, comps_pre=4,
         age=21, peak_age=24, age_vs_peak=-3, seasons_elite=4,
         prior_olympics=0, first_olympics=1, win_streak=11,
         cv_pre=0.032, peak_proximity=1.038, consistency_load=2.45,
         olympic_exp_sq=0, recency_decay=0.65, ascending=1, elite_log=1.58,
         sentiment=0.24, actual_score=61.0),

    # ── DOMINANT_EXP ──────────────────────────────────────────
    # Pattern: top pre-Games scores, prior Olympic experience
    # Characteristic outcome: performs at or near prediction
    # Preparation complete — experience closes the gap
    dict(name='DOMINANT_EXP', games='Rio_2016', discipline='Gymnastics AA',
         mean_pre=94.8, std_pre=2.2, best_pre=97.6, comps_pre=5,
         age=22, peak_age=23, age_vs_peak=-1, seasons_elite=6,
         prior_olympics=1, first_olympics=0, win_streak=5,
         cv_pre=0.023, peak_proximity=1.026, consistency_load=1.65,
         olympic_exp_sq=1, recency_decay=0.63, ascending=1, elite_log=1.92,
         sentiment=0.71, actual_score=96.8),

    # Second instance — past peak but deeply experienced
    dict(name='DOMINANT_EXP', games='Tokyo_2021', discipline='Swimming Distance',
         mean_pre=87.2, std_pre=4.4, best_pre=91.8, comps_pre=5,
         age=31, peak_age=26, age_vs_peak=+5, seasons_elite=16,
         prior_olympics=4, first_olympics=0, win_streak=1,
         cv_pre=0.050, peak_proximity=0.993, consistency_load=0.0,
         olympic_exp_sq=16, recency_decay=0.64, ascending=0, elite_log=2.83,
         sentiment=0.73, actual_score=88.5),

    # ── ELITE_RETURN ──────────────────────────────────────────
    # Pattern: elite athlete returning to competition after adversity
    # Characteristic outcome: full performance restoration when support is adequate
    # High variance — outcome depends heavily on preparation investment
    dict(name='ELITE_RETURN', games='Paris_2024', discipline='Gymnastics Floor',
         mean_pre=96.0, std_pre=1.6, best_pre=97.5, comps_pre=4,
         age=27, peak_age=22, age_vs_peak=+5, seasons_elite=11,
         prior_olympics=2, first_olympics=0, win_streak=4,
         cv_pre=0.017, peak_proximity=0.983, consistency_load=1.18,
         olympic_exp_sq=4, recency_decay=0.70, ascending=1, elite_log=2.42,
         sentiment=0.67, actual_score=95.1),

    # ── VOLATILE ──────────────────────────────────────────────
    # Pattern: high pre-Games average, high variance, inconsistent results
    # Characteristic outcome: medal or DNF — rarely middle
    # Investment in consistency training most valuable here
    dict(name='VOLATILE', games='Tokyo_2021', discipline='Swimming Distance',
         mean_pre=87.8, std_pre=8.2, best_pre=93.5, comps_pre=5,
         age=28, peak_age=26, age_vs_peak=+2, seasons_elite=10,
         prior_olympics=3, first_olympics=0, win_streak=0,
         cv_pre=0.093, peak_proximity=0.998, consistency_load=0.0,
         olympic_exp_sq=9, recency_decay=0.59, ascending=0, elite_log=2.40,
         sentiment=0.51, actual_score=0.0),   # DNF

    # ── ASCENDING ─────────────────────────────────────────────
    # Pattern: program on upward trajectory, recent breakout result
    # Investment thesis: development funding, higher variance, higher ceiling
    # LA 2028 analog — program that just produced a breakout Games result
    dict(name='ASCENDING', games='Paris_2024', discipline='Gymnastics AA',
         mean_pre=78.4, std_pre=5.6, best_pre=84.1, comps_pre=4,
         age=19, peak_age=22, age_vs_peak=-3, seasons_elite=3,
         prior_olympics=0, first_olympics=1, win_streak=2,
         cv_pre=0.071, peak_proximity=1.073, consistency_load=0.0,
         olympic_exp_sq=0, recency_decay=0.52, ascending=1, elite_log=1.20,
         sentiment=0.58, actual_score=82.3),

    # ── MAINTENANCE ───────────────────────────────────────────
    # Pattern: perennially dominant program, medal near-certain
    # Investment thesis: protect the floor, not expand the ceiling
    # Marginal return on additional investment is low — already near optimal
    dict(name='MAINTENANCE', games='Rio_2016', discipline='Soccer Women',
         mean_pre=91.5, std_pre=2.8, best_pre=94.2, comps_pre=6,
         age=26, peak_age=25, age_vs_peak=+1, seasons_elite=8,
         prior_olympics=2, first_olympics=0, win_streak=3,
         cv_pre=0.031, peak_proximity=0.996, consistency_load=0.82,
         olympic_exp_sq=4, recency_decay=0.68, ascending=0, elite_log=2.18,
         sentiment=0.69, actual_score=92.1),
])

print(f'Training data: {len(training_data)} composite archetype rows')
print()
print(training_data[["name","games","mean_pre","std_pre","first_olympics",
                      "win_streak","sentiment","actual_score"]].to_string(index=False))
print()
print("Archetypes represent cross-athlete, cross-cycle patterns.")
print("DOMINANT_FIRST: highest preparation gap signal — key LP portfolio flag.")
print("ASCENDING: development investment thesis — post-breakout Games program.")
print("MAINTENANCE: protection investment — near-certain medal, low marginal return.")


In [ ]:
# ── NLP sentiment scoring ─────────────────────────────────────

burden_markers = [
    'pressure', 'burden', 'weight', 'invisible', 'darkness', 'fear',
    'hatred', 'noise', 'expectations', 'insurmountable', 'overwhelming',
    'tainted', 'battles', 'survive', 'block out', 'immense', 'exhausted'
]
confidence_markers = [
    'excited', 'ready', 'trust', 'enjoy', 'fun', 'joy', 'love',
    'good place', 'prepared', 'myself', 'for me', 'different', 'strong'
]

def score_athlete_statements(statements_list):
    """Score a list of pre-Games statements for preparation readiness."""
    if not statements_list:
        return 0.0
    text = ' '.join(statements_list).lower()
    blob = TextBlob(text)
    base  = blob.sentiment.polarity
    burden = sum(1 for m in burden_markers if m in text) / len(burden_markers)
    conf   = sum(1 for m in confidence_markers if m in text) / len(confidence_markers)
    return round(base + conf - burden * 1.5, 3)


# Pre-Games statements — add as you collect them
# Source: Olympics.com interviews, official press conferences, athlete social media
pre_games_statements = {
    'DOMINANT_FIRST': [
        "On the world's biggest stage, those who appear the strongest may still be fighting invisible battles.",
        "Vile online hatred attacks the mind and fear lures it into the darkness.",
        "The endless insurmountable pressure.",
        "So many eyes, so much attention. It really can get to you.",
        "I was not ready to handle that to a full extent.",
    ],
    'DOMINANT_EXP': [
        "I feel really good about where I am right now.",
        "I trust my preparation completely.",
        "This is my fourth Olympics and I just want to enjoy every moment.",
        "I'm ready to compete and I'm excited to be here.",
    ],
    'MAINTENANCE': [
        "I feel strong this season.",
        "I've been preparing for this my whole life.",
        "I'm excited to be at my first Olympics.",
        "Just going to go out and skate my race.",
    ],
}

# Score each athlete and add to athletes_df
athletes_df['sentiment'] = athletes_df['name'].map(
    {name: score_athlete_statements(stmts)
     for name, stmts in pre_games_statements.items()}
).fillna(0.3)  # default neutral-positive for athletes without statements

print('NLP sentiment scores:')
for name, score in sorted(
    {n: score_athlete_statements(s) for n, s in pre_games_statements.items()}.items(),
    key=lambda x: x[1]
):
    print(f'  {name:<25} {score:+.3f}')


In [ ]:
# ── Gradient boosting + quantile regression ───────────────────

FEATURE_COLS = [
    'mean_pre', 'std_pre', 'best_pre', 'comps_pre',
    'age_vs_peak', 'cv_pre', 'peak_proximity',
    'first_olympics', 'olympic_exp_sq', 'win_streak',
    'consistency_load', 'recency_decay', 'ascending',
    'elite_log', 'sentiment'
]

train = training_data.dropna(subset=FEATURE_COLS + ['actual_score'])
X_train = train[FEATURE_COLS].values
y_train = train['actual_score'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

# ── Gradient boosting — point prediction ──────────────────────
lgb_params = dict(n_estimators=50, max_depth=2, learning_rate=0.05,
                  min_child_samples=2, reg_alpha=1.0, reg_lambda=2.0,
                  subsample=0.8, colsample_bytree=0.7,
                  random_state=42, verbose=-1)
model_gbm = lgb.LGBMRegressor(**lgb_params)

# LOO CV for honest error estimate
loo = LeaveOneOut()
loo_preds = np.zeros(len(y_train))
for tr_idx, te_idx in loo.split(X_scaled):
    m = lgb.LGBMRegressor(**lgb_params)
    m.fit(X_scaled[tr_idx], y_train[tr_idx])
    loo_preds[te_idx] = m.predict(X_scaled[te_idx])

model_gbm.fit(X_scaled, y_train)

mae_loo  = mean_absolute_error(y_train, loo_preds)
rmse_loo = np.sqrt(mean_squared_error(y_train, loo_preds))

# ── Quantile regression — uncertainty intervals ───────────────
q_models = {}
for q in [0.10, 0.50, 0.90]:
    qr = QuantileRegressor(quantile=q, alpha=0.5, solver='highs')
    qr.fit(X_scaled, y_train)
    q_models[q] = qr

print('=== Performance Prediction Model ===')
print(f'Training N:   {len(train)}')
print(f'LOO MAE:      {mae_loo:.1f} score points')
print(f'LOO RMSE:     {rmse_loo:.1f} score points')
print()

# Feature importance
fi = pd.DataFrame({'feature': FEATURE_COLS,
                   'importance': model_gbm.feature_importances_}
                  ).sort_values('importance', ascending=False)
print('Top 8 features:')
print(fi.head(8).to_string(index=False))


In [ ]:
# ── Add predictions to athletes_df ────────────────────────────

def predict_athlete(row, model, q_models, scaler, feature_cols):
    """Predict mean and std for a single athlete row."""
    try:
        x = scaler.transform([row[feature_cols].values.astype(float)])
        pred_mean = float(model.predict(x)[0])
        p10 = float(q_models[0.10].predict(x)[0])
        p90 = float(q_models[0.90].predict(x)[0])
        pred_std  = max((p90 - p10) / (2 * 1.28), 1.0)
        return pred_mean, pred_std
    except Exception:
        return row.get('mean_pre', 80.0), row.get('std_pre', 5.0)


if not athletes_df.empty:
    pred_means, pred_stds = [], []
    for _, row in athletes_df.iterrows():
        # Fill missing features with defaults
        for col in FEATURE_COLS:
            if col not in row or pd.isna(row.get(col)):
                row = row.copy()
                row[col] = 0.0
        pm, ps = predict_athlete(row, model_gbm, q_models, scaler, FEATURE_COLS)

        # NLP adjustment: low sentiment → wider std
        nlp_adj  = max(1.0, 1.0 + (0.3 - row.get('sentiment', 0.3)) * 0.5)
        pred_means.append(round(pm, 1))
        pred_stds.append(round(ps * nlp_adj, 2))

    athletes_df['pred_mean'] = pred_means
    athletes_df['pred_std']  = pred_stds
    athletes_df['nlp_adj']   = [
        round(max(1.0, 1.0 + (0.3 - s) * 0.5), 3)
        for s in athletes_df.get('sentiment', pd.Series([0.3]*len(athletes_df)))
    ]

    print('✅ Predictions added to athletes_df')
    print()
    cols = ['name','mean_pre','std_pre','pred_mean','pred_std','nlp_adj']
    available = [c for c in cols if c in athletes_df.columns]
    print(athletes_df[available].to_string(index=False))
else:
    print('athletes_df empty — populate Section 0 with real data')
    athletes_df['pred_mean'] = athletes_df.get('mean_pre', pd.Series(dtype=float))
    athletes_df['pred_std']  = athletes_df.get('std_pre', pd.Series(dtype=float))


---
## Section 3 — Monte Carlo Simulation

Uses `pred_mean` and `pred_std` from Section 2 instead of naive historical scores.  
Computes P(gold), P(medal), P(top5) for each athlete.


In [ ]:
# ── Competitor fields ─────────────────────────────────────────
# International competitor distributions by event
# Source: same pre-Olympic season data from Section 0
# (mean, std) pairs for non-USA competitors

competitor_fields = {
    'Figure Skating Men': [
        (89.5, 4.2),   # VOLATILE-level
        (87.2, 3.8),   # DOMINANT_EXP-level
        (85.9, 4.5),   # Sato-level
        (84.1, 5.1),
        (82.3, 5.8),
    ],
    'Figure Skating Women': [
        (88.4, 3.9),   # DOMINANT_EXP-level
        (86.1, 4.2),
        (84.5, 5.0),
    ],
    'Speed Skating M 500m': [
        (91.8, 2.1),   # Dubreuil-level
        (90.4, 2.8),
        (89.1, 3.2),
        (88.0, 3.5),
    ],
    'Alpine Women SL': [
        (87.2, 8.4),   # Vlhova-level
        (84.1, 9.1),
        (82.3, 9.8),
    ],
    'Athletics Men 100m': [
        (91.2, 2.8),
        (90.5, 3.1),
        (89.8, 3.4),
        (89.1, 3.8),
    ],
}


def compute_p_gold(pred_mean, pred_std, competitor_params, n_sims=N_SIMS):
    usa  = np.random.normal(pred_mean, max(pred_std, 0.5), n_sims)
    intl = np.column_stack([
        np.random.normal(m, s, n_sims) for m, s in competitor_params
    ])
    rank = (intl >= usa[:, None]).sum(axis=1) + 1
    return (
        round((rank == 1).mean() * 100, 1),
        round((rank <= 3).mean() * 100, 1),
        round((rank <= 5).mean() * 100, 1),
    )


if not athletes_df.empty:
    p_golds, p_medals, p_top5s = [], [], []
    for _, row in athletes_df.iterrows():
        disc  = row.get('discipline', '')
        field = competitor_fields.get(disc, [(80.0, 5.0)] * 5)
        pm    = row.get('pred_mean', row.get('mean_pre', 80.0))
        ps    = row.get('pred_std',  row.get('std_pre',  5.0))
        pg, pm_medal, pt = compute_p_gold(pm, ps, field)
        p_golds.append(pg)
        p_medals.append(pm_medal)
        p_top5s.append(pt)

    athletes_df['p_gold']  = p_golds
    athletes_df['p_medal'] = p_medals
    athletes_df['p_top5']  = p_top5s

    print('✅ Monte Carlo complete')
    print()
    cols = ['name','discipline','pred_mean','pred_std','p_gold','p_medal']
    available = [c for c in cols if c in athletes_df.columns]
    print(athletes_df[available].sort_values('p_gold', ascending=False).to_string(index=False))


---
## Section 4 — LP/IP Optimization

**IP** for binary athlete selection (fund or don't fund).  
**LP** for efficient frontier sweep (40 budget levels, ~100x faster).  
Cross-Games LP for multi-cycle budget allocation.


In [ ]:
# ── Athlete cost and qualification slots ──────────────────────
# cost: normalized investment unit (1.0 = standard full support)
# qual_slot: True if athlete has qualified / is likely to qualify

cost_map = {
    'Figure Skating Men':   1.2,
    'Figure Skating Women': 1.0,
    'Speed Skating M 500m': 0.8,
    'Alpine Women SL':      1.1,
    'Athletics Men 100m':   0.9,
}

if not athletes_df.empty:
    athletes_df['cost']      = athletes_df['discipline'].map(cost_map).fillna(1.0)
    athletes_df['qual_slot'] = True

BUDGET        = 8.0
SPORT_MINS    = {'Figure Skating': 1}
MAX_PER_EVENT = 3


# ── IP solver ─────────────────────────────────────────────────
def solve_ip(df, budget, sport_mins, max_per_event, min_gold=0.0):
    eligible = df[
        df.get('qual_slot', pd.Series([True]*len(df))) &
        df['p_gold'].notna() &
        (df['p_gold'] >= min_gold)
    ].copy()
    if eligible.empty:
        return None, 0.0, 'no_eligible'

    prob = pulp.LpProblem('USOPC_IP', pulp.LpMaximize)
    x = {i: pulp.LpVariable(f'x{i}', cat='Binary') for i in eligible.index}
    prob += pulp.lpSum((eligible.loc[i,'p_gold']/100) * x[i] for i in eligible.index)
    prob += (pulp.lpSum(eligible.loc[i,'cost'] * x[i] for i in eligible.index) <= budget)

    for sport, mn in sport_mins.items():
        idxs = [i for i in eligible.index
                if sport.lower() in eligible.loc[i,'discipline'].lower()]
        if idxs and mn > 0:
            prob += (pulp.lpSum(x[i] for i in idxs) >= mn)

    for ev in eligible['discipline'].unique():
        idxs = [i for i in eligible.index if eligible.loc[i,'discipline'] == ev]
        if len(idxs) > max_per_event:
            prob += (pulp.lpSum(x[i] for i in idxs) <= max_per_event)

    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    status = pulp.LpStatus[prob.status]
    if status != 'Optimal':
        return None, 0.0, status

    sel = eligible.loc[[i for i in eligible.index if pulp.value(x[i]) > 0.5]].copy()
    exp = float(pulp.value(prob.objective))
    return sel, exp, status


# ── LP solver with shadow prices ──────────────────────────────
#
# Shadow price (dual variable) on the budget constraint answers:
# "What is one additional unit of budget worth in expected medals?"
#
# High shadow price = you are budget constrained, more funding changes outcomes
# Low shadow price  = you have funded everything worth funding
#
# Shadow price of NOT winning:
# For each non-selected athlete, their reduced cost (rc) is the gap between
# their value to the objective and what it would cost to include them.
# A large negative rc means they are far from being worth selecting.
# A small negative rc means they are just below the cut — marginal athletes
# who would enter the portfolio with a small budget increase.
#
# This captures the asymmetric political cost:
# Not funding a gold medalist hurts more than funding a non-medalist.
# Athletes with small rc are the ones highest support need of being "the one we not yet reached."

def solve_lp_with_shadow(df, budget, sport_mins, max_per_event, min_gold=0.0):
    """
    LP solver that returns shadow prices on constraints and reduced costs per athlete.
    shadow_budget:  marginal value of one more budget unit (expected golds per unit)
    reduced_costs:  per-athlete gap from selection threshold
                    negative = not selected, magnitude = how far below threshold
                    near-zero negative = "almost selected" — highest portfolio exposure
    """
    eligible = df[
        df.get('qual_slot', pd.Series([True]*len(df))) &
        df['p_gold'].notna() &
        (df['p_gold'] >= min_gold)
    ].copy()
    if eligible.empty:
        return None, 0.0, 0.0, {}, {}, 'no_eligible'

    prob = pulp.LpProblem('USOPC_LP_SHADOW', pulp.LpMaximize)
    x = {i: pulp.LpVariable(f'x{i}', lowBound=0, upBound=1) for i in eligible.index}
    prob += pulp.lpSum((eligible.loc[i,'p_gold']/100) * x[i] for i in eligible.index)

    # Named budget constraint — needed to extract shadow price
    budget_constraint = pulp.lpSum(
        eligible.loc[i,'cost'] * x[i] for i in eligible.index
    ) <= budget
    prob += budget_constraint, 'budget'

    for sport, mn in sport_mins.items():
        idxs = [i for i in eligible.index
                if sport.lower() in eligible.loc[i,'discipline'].lower()]
        if idxs and mn > 0:
            prob += (pulp.lpSum(x[i] for i in idxs) >= mn), f'min_{sport}'

    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    status = pulp.LpStatus[prob.status]
    if status != 'Optimal':
        return None, 0.0, 0.0, {}, {}, status

    lp_obj = float(pulp.value(prob.objective))

    # ── Shadow price on budget constraint ─────────────────────
    # pi = dual variable = marginal expected golds per budget unit
    shadow_budget = 0.0
    for name, constraint in prob.constraints.items():
        if name == 'budget':
            shadow_budget = -constraint.pi if constraint.pi is not None else 0.0

    # ── Reduced costs per athlete ──────────────────────────────
    # rc > 0: athlete is in solution at upper bound (fully funded)
    # rc < 0: athlete is out of solution, magnitude = gap from threshold
    # rc near 0 and negative: just not yet reached — highest "not yet reached gold" portfolio exposure
    reduced_costs = {}
    for i in eligible.index:
        rc = x[i].dj if x[i].dj is not None else 0.0
        reduced_costs[i] = round(rc, 4)

    # Greedy rounding for integer selection
    srt = eligible.sort_values('p_gold', ascending=False)
    sel_idx, cost_used = [], 0.0
    for i, row in srt.iterrows():
        if cost_used + row['cost'] <= budget + 1e-6:
            sel_idx.append(i)
            cost_used += row['cost']

    sel = eligible.loc[sel_idx].copy()
    exp = (sel['p_gold'] / 100).sum()

    return sel, exp, lp_obj, shadow_budget, reduced_costs, status


# ── Efficient frontier with shadow prices ─────────────────────
def efficient_frontier(df, sport_mins, max_per_event, steps=40):
    max_b = df['cost'].sum()
    records, prev_g, prev_b = [], 0.0, 0.0
    for b in np.linspace(0, max_b, steps):
        sel, exp, lp_obj, shadow, rc, st = solve_lp_with_shadow(
            df, b, sport_mins, max_per_event
        )
        if st == 'Optimal' and sel is not None:
            dg = exp - prev_g
            db = b  - prev_b
            records.append({
                'budget':         round(b, 2),
                'exp_golds':      round(exp, 4),
                'lp_upper_bound': round(lp_obj, 4),
                'rounding_gap':   round(lp_obj - exp, 4),
                'n_athletes':     len(sel),
                'marginal_cost':  round(db/dg, 3) if dg > 0.001 else None,
                'shadow_price':   round(shadow, 4),
                # Shadow price interpretation:
                # High = budget is binding, more money changes outcomes
                # Low  = diminishing returns, portfolio is near-complete
            })
            prev_g, prev_b = exp, b
    return pd.DataFrame(records)


# ── Run optimization ──────────────────────────────────────────
if not athletes_df.empty and 'p_gold' in athletes_df.columns:

    # IP for optimal portfolio
    print('=== IP Optimal Portfolio ===')
    sel, exp_golds, status = solve_ip(athletes_df, BUDGET, SPORT_MINS, MAX_PER_EVENT)
    if sel is not None:
        print(f'Status: {status} | Budget: {BUDGET} | E[Golds]: {exp_golds:.3f}')
        print()
        print(sel[['name','discipline','p_gold','p_medal','cost']].sort_values(
            'p_gold', ascending=False).to_string(index=False))
        athletes_df['selected'] = athletes_df.index.isin(sel.index).astype(int)
    else:
        print('No optimal solution — check data')
        athletes_df['selected'] = 0

    # LP with shadow prices at current budget
    print()
    print('=== Shadow Price Analysis ===')
    _, exp_lp, lp_obj, shadow_budget, reduced_costs, st = solve_lp_with_shadow(
        athletes_df, BUDGET, SPORT_MINS, MAX_PER_EVENT
    )

    print(f'Budget shadow price: {shadow_budget:.4f} expected golds per budget unit')
    print()
    if shadow_budget > 0.05:
        print('  Interpretation: BUDGET IS BINDING')
        print(f'  One more unit of budget adds {shadow_budget:.3f} expected golds')
        print('  Case for increased USOPC investment is strong')
    elif shadow_budget > 0.01:
        print('  Interpretation: MODERATE BUDGET PRESSURE')
        print('  Marginal returns on additional investment are positive but declining')
    else:
        print('  Interpretation: BUDGET NOT BINDING')
        print('  Portfolio is near-complete — additional funding adds little')

    # Reduced costs — portfolio exposure of not yet reached medals
    print()
    print('=== Reduced Costs — Political Risk of Missed Medals ===')
    print('Athletes NOT selected, ranked by proximity to selection threshold')
    print('Near-zero = almost selected = highest risk of being "the one we not yet reached"')
    print()

    rc_df = athletes_df.copy()
    rc_df['reduced_cost'] = rc_df.index.map(reduced_costs).fillna(0.0)
    rc_df['selected_flag'] = rc_df.index.isin(sel.index if sel is not None else [])

    not_selected = rc_df[~rc_df['selected_flag']].copy()
    not_selected = not_selected.sort_values('reduced_cost', ascending=False)

    if not not_selected.empty:
        cols = ['name','discipline','p_gold','p_medal','cost','reduced_cost']
        avail = [c for c in cols if c in not_selected.columns]
        print(not_selected[avail].to_string(index=False))
        print()

        # Most politically at-risk athlete
        most_at_risk = not_selected.iloc[0]
        print(f'Highest portfolio exposure: {most_at_risk["name"]}')
        print(f'  P(gold): {most_at_risk["p_gold"]:.1f}%')
        print(f'  Reduced cost: {most_at_risk["reduced_cost"]:.4f}')
        print(f'  Interpretation: if this athlete wins gold without USOPC support,')
        print(f'  that is the most defensible omission to explain.')

    # Efficient frontier
    print()
    print('Computing efficient frontier with shadow prices (LP, 40 steps)...')
    frontier_df = efficient_frontier(athletes_df, SPORT_MINS, MAX_PER_EVENT)
    print(f'Frontier points: {len(frontier_df)}')
    if not frontier_df.empty:
        print()
        print(frontier_df[['budget','exp_golds','shadow_price',
                            'marginal_cost','n_athletes']].tail(8).to_string(index=False))
        print()
        print('shadow_price column: marginal value of one budget unit at each frontier point')
        print('Read down the column to see where returns start diminishing')

else:
    print('Skipping optimization — populate athletes_df with real data first')
    athletes_df['selected'] = 0
    frontier_df = pd.DataFrame()


---
## Section 5 — Policy Simulation

Investment → performance feedback loop.  
Models what happens to P(gold) when USOPC funds specific support programs.  
Key question: would stress inoculation have changed the outcome for a DOMINANT_FIRST Summer Games athlete — e.g. dominant pre-Games record, Paris 2024 as first real Olympic moment?


In [ ]:
# ── Investment response functions ─────────────────────────────

def apply_investment_to_athlete(row, investments):
    """
    Apply investment vector to an athlete row.
    Returns updated row with adjusted pred_mean and pred_std.
    investments: dict of {category: level} where level is 0-1
    """
    row = row.copy()

    psych_inv  = investments.get('sports_psychology', 0)
    stress_inv = investments.get('stress_inoculation', 0)
    coach_inv  = investments.get('elite_coaching', 0)
    period_inv = investments.get('periodization', 0)

    # Psychological load reduction → mean score improvement
    base_load = max(0, 0.5 - row.get('sentiment', 0.0))
    load_reduction = 0.4 * (1 - np.exp(-3 * psych_inv))
    psych_gain = load_reduction * base_load * 15
    row['pred_mean'] = row.get('pred_mean', row.get('mean_pre', 80)) + psych_gain

    # Stress inoculation → reduces first-Olympics variance
    if row.get('first_olympics', 0):
        sip = stress_inv * 0.7
        row['pred_std'] = row.get('pred_std', row.get('std_pre', 5)) * (1 - sip * 0.5)

    # Periodization → tighter std
    period_reduction = 0.25 * period_inv
    row['pred_std'] = row.get('pred_std', 5) * (1 - period_reduction)

    return row


# ── Run scenarios for key athletes ───────────────────────────
investment_scenarios = {
    'Baseline':                       {},
    'Sports Psychology':              {'sports_psychology': 0.8},
    'Stress Inoculation':             {'stress_inoculation': 0.9},
    'Psych + Stress':                 {'sports_psychology': 0.7, 'stress_inoculation': 0.9},
    'Full Protocol':                  {'sports_psychology': 0.8, 'stress_inoculation': 0.9,
                                       'elite_coaching': 0.6, 'periodization': 0.5},
}

if not athletes_df.empty and 'pred_mean' in athletes_df.columns:
    # Focus on DOMINANT_FIRST as the key case
    malinin_row = athletes_df[athletes_df['name'] == 'DOMINANT_FIRST']

    if not malinin_row.empty:
        malinin_row = malinin_row.iloc[0]
        print('=== DOMINANT_FIRST 2026 — Investment Scenarios ===')
        print()

        scenario_results = []
        for scenario_name, investments in investment_scenarios.items():
            updated = apply_investment_to_athlete(malinin_row, investments)
            disc = malinin_row.get('discipline', 'Figure Skating Men')
            field = competitor_fields.get(disc, [(85.0, 4.0)] * 5)
            pg, pm_medal, _ = compute_p_gold(
                updated['pred_mean'], updated['pred_std'], field
            )
            cost = sum(
                investments.get(c, 0) * {'sports_psychology': 0.8, 'stress_inoculation': 0.6,
                                          'elite_coaching': 1.5, 'periodization': 0.4}.get(c, 1.0)
                for c in investments
            )
            scenario_results.append({
                'scenario':  scenario_name,
                'pred_mean': round(updated['pred_mean'], 1),
                'pred_std':  round(updated['pred_std'], 2),
                'p_gold':    pg,
                'p_medal':   pm_medal,
                'cost':      round(cost, 2)
            })
            print(f'{scenario_name:<28} P(gold): {pg:5.1f}%  '
                  f'P(medal): {pm_medal:5.1f}%  Mean: {updated["pred_mean"]:.1f}  '
                  f'Std: {updated["pred_std"]:.2f}  Cost: {cost:.2f}')

        scenario_df = pd.DataFrame(scenario_results)
    else:
        print('DOMINANT_FIRST not found in athletes_df — check Section 0 data')
        scenario_df = pd.DataFrame()
else:
    print('Skipping policy simulation — run Section 2 first')
    scenario_df = pd.DataFrame()


---
## Section 6 — Validation & Backtesting

Score every forecast against actual results.  
**Brier score** for probability calibration.  
**MAE / RMSE** for point prediction accuracy.  
**Skill score** for value added over naive baseline.

Populate `actual_results` after each Games.


In [ ]:
# ── Actual results — COMPOSITE ARCHETYPE ground truth ─────────
# Results map to archetype patterns, not individuals.
# Private repo contains named athlete results.

actual_results = pd.DataFrame([
    # Paris_2024 — DOMINANT_FIRST, underperformed
    dict(name='DOMINANT_FIRST', games='Paris_2024', discipline='Gymnastics AA',
         actual_score=61.0,  actual_gold=0, actual_medal=0),

    # Rio_2016 — DOMINANT_EXP, gold
    dict(name='DOMINANT_EXP',   games='Rio_2016', discipline='Gymnastics AA',
         actual_score=96.8,  actual_gold=1, actual_medal=1),

    # Tokyo_2021 — DOMINANT_EXP, gold
    dict(name='DOMINANT_EXP',   games='Tokyo_2021', discipline='Swimming Distance',
         actual_score=88.5,  actual_gold=1, actual_medal=1),

    # Tokyo_2021 — VOLATILE, DNF
    dict(name='VOLATILE',       games='Tokyo_2021', discipline='Swimming Distance',
         actual_score=0.0,   actual_gold=0, actual_medal=0),

    # Paris_2024 — ELITE_RETURN, gold
    dict(name='ELITE_RETURN',   games='Paris_2024', discipline='Gymnastics Floor',
         actual_score=95.1,  actual_gold=1, actual_medal=1),

    # Tokyo_2021 — DOMINANT_FIRST, underperformed
    dict(name='DOMINANT_FIRST', games='Tokyo_2021', discipline='Gymnastics AA',
         actual_score=63.8,  actual_gold=0, actual_medal=0),

    # Paris_2024 — ASCENDING, bronze
    dict(name='ASCENDING',      games='Paris_2024', discipline='Gymnastics AA',
         actual_score=82.3,  actual_gold=0, actual_medal=1),
])

# ── Merge with training predictions ───────────────────────────
val_df = training_data.copy()
val_df = val_df.merge(
    actual_results[['name','games','actual_score','actual_gold','actual_medal']],
    on=['name','games'], suffixes=('_pred', '_actual')
)

# Use actual_score_actual as ground truth
if 'actual_score_actual' in val_df.columns:
    val_df['actual_score'] = val_df['actual_score_actual']
elif 'actual_score' in val_df.columns:
    pass  # already correct

# Compute LOO predictions for validation
if len(val_df) >= 3:
    X_val = val_df[FEATURE_COLS].fillna(0).values
    y_val = val_df['actual_score'].values
    X_val_scaled = scaler.transform(X_val)

    loo_val = np.zeros(len(y_val))
    for tr_idx, te_idx in LeaveOneOut().split(X_val_scaled):
        m = lgb.LGBMRegressor(**lgb_params)
        m.fit(X_val_scaled[tr_idx], y_val[tr_idx])
        loo_val[te_idx] = m.predict(X_val_scaled[te_idx])

    val_df['pred_score_nb2'] = loo_val.round(1)
    val_df['naive_pred']     = val_df['mean_pre']

    # Errors
    val_df['nb2_error']  = val_df['pred_score_nb2'] - val_df['actual_score']
    val_df['naive_error']= val_df['naive_pred']      - val_df['actual_score']
    val_df['nb2_abs_err']  = val_df['nb2_error'].abs()
    val_df['naive_abs_err']= val_df['naive_error'].abs()

    nb2_mae   = val_df['nb2_abs_err'].mean()
    naive_mae = val_df['naive_abs_err'].mean()
    nb2_rmse  = np.sqrt((val_df['nb2_error']**2).mean())
    naive_rmse= np.sqrt((val_df['naive_error']**2).mean())

    print('=== Validation Results ===')
    print()
    print(f'{"Metric":<20} {"Naive (historical mean)":<28} {"NB2 (model)":<20} Improvement')
    print('-' * 80)
    print(f'{"MAE (pts)":<20} {naive_mae:<28.2f} {nb2_mae:<20.2f} {(naive_mae-nb2_mae)/naive_mae*100:.1f}%')
    print(f'{"RMSE (pts)":<20} {naive_rmse:<28.2f} {nb2_rmse:<20.2f} {(naive_rmse-nb2_rmse)/naive_rmse*100:.1f}%')
    print()

    # Brier scores if actual_gold available
    if 'actual_gold' in val_df.columns:
        y_bin = val_df['actual_gold'].values.astype(float)
        base_rate = y_bin.mean()
        naive_brier_score = brier_score_loss(y_bin, np.full_like(y_bin, base_rate))
        # Use mean_pre normalized as naive probability proxy
        naive_p = np.clip(val_df['mean_pre'] / 100, 0.01, 0.99)
        nb1_brier_score = brier_score_loss(y_bin, naive_p)
        nb1_bss = 1 - nb1_brier_score / (naive_brier_score + 1e-9)

        print(f'Naive Brier Score:  {naive_brier_score:.4f}')
        print(f'NB1  Brier Score:   {nb1_brier_score:.4f}')
        print(f'Brier Skill Score:  {nb1_bss:.4f}  (positive = beats naive)')
        print()

    print('Per-athlete errors:')
    cols = ['name','games','actual_score','naive_pred','naive_abs_err',
            'pred_score_nb2','nb2_abs_err']
    avail = [c for c in cols if c in val_df.columns]
    print(val_df[avail].sort_values('nb2_abs_err', ascending=False).to_string(index=False))
else:
    print('Not enough validation records — add more historical data')
    nb2_mae = naive_mae = 0
    nb2_rmse = naive_rmse = 0
    nb1_bss = 0


In [ ]:
# ── Rank error and Spearman correlation ──────────────────────
#
# Score error tells you how far off the absolute prediction was.
# Rank error tells you how far off the competitive placement was.
# Spearman rho tells you whether the ordering was roughly right.
#
# The high-preparation-gap case (DOMINANT_FIRST, Games_2026) is the key validation:
#   Projected rank: 1st (highest pred_mean in field)
#   Actual rank:    8th
#   Rank error:     +7 (we overestimated by 7 places)

from scipy.stats import spearmanr

# ── Full event results — needed to compute rank ───────────────
# One row per athlete per event — both USA and international
# actual_rank: where they actually finished
# pred_rank:   where our model predicted they would finish
#              (rank by pred_mean descending within event)

full_event_results = pd.DataFrame([
    # Games_2026 — Men's Figure Skating — full field
    # actual_rank from official results, pred_score from model
    dict(name='VOLATILE', event='Figure Skating Men', games='Games_2026',
         pred_score=89.5, actual_score=289.41, actual_rank=1),
    dict(name='DOMINANT_EXP',     event='Figure Skating Men', games='Games_2026',
         pred_score=87.2, actual_score=278.33, actual_rank=2),
    dict(name='DEVELOPMENT',         event='Figure Skating Men', games='Games_2026',
         pred_score=85.9, actual_score=275.18, actual_rank=3),
    dict(name='DOMINANT_FIRST',      event='Figure Skating Men', games='Games_2026',
         pred_score=97.8, actual_score=234.82, actual_rank=8),  # THE KEY CASE

    # Games_2022 — Men's Figure Skating
    dict(name='DOMINANT_EXP',       event='Figure Skating Men', games='Games_2022',
         pred_score=95.4, actual_score=332.60, actual_rank=1),
    dict(name='DOMINANT_EXP',     event='Figure Skating Men', games='Games_2022',
         pred_score=87.2, actual_score=310.05, actual_rank=2),
    dict(name='Shoma Uno',         event='Figure Skating Men', games='Games_2022',
         pred_score=85.0, actual_score=293.00, actual_rank=3),

    # Games_2018 — Men's Figure Skating
    dict(name='Yuzuru Hanyu',      event='Figure Skating Men', games='Games_2018',
         pred_score=97.2, actual_score=317.85, actual_rank=1),
    dict(name='Shoma Uno',         event='Figure Skating Men', games='Games_2018',
         pred_score=85.0, actual_score=306.90, actual_rank=2),
    dict(name='DOMINANT_EXP',       event='Figure Skating Men', games='Games_2018',
         pred_score=82.1, actual_score=297.35, actual_rank=5),  # 17th in SP, 1st in FS
])

# ── Compute predicted rank within each event/games ────────────
# pred_rank: rank by pred_score descending (highest predicted score = rank 1)
full_event_results['pred_rank'] = (
    full_event_results
    .groupby(['event', 'games'])['pred_score']
    .rank(ascending=False)
    .astype(int)
)

# ── Rank error ─────────────────────────────────────────────────
full_event_results['rank_error']     = full_event_results['pred_rank'] - full_event_results['actual_rank']
full_event_results['abs_rank_error'] = full_event_results['rank_error'].abs()

# ── Score error for comparison ────────────────────────────────
# Normalize actual_score to same 0-100 scale as pred_score for comparison
for (event, games), grp in full_event_results.groupby(['event', 'games']):
    max_s = grp['actual_score'].max()
    full_event_results.loc[grp.index, 'actual_score_norm'] = (
        grp['actual_score'] / max_s * 100
    )

full_event_results['score_error']     = full_event_results['pred_score'] - full_event_results['actual_score_norm']
full_event_results['abs_score_error'] = full_event_results['score_error'].abs()

print('=== Rank Error Analysis ===')
print()
print(full_event_results[
    ['name','games','pred_score','pred_rank','actual_rank',
     'rank_error','abs_rank_error','abs_score_error']
].sort_values(['games','actual_rank']).to_string(index=False))
print()

# ── Summary statistics ────────────────────────────────────────
mean_abs_rank_err  = full_event_results['abs_rank_error'].mean()
mean_abs_score_err = full_event_results['abs_score_error'].mean()
median_rank_err    = full_event_results['abs_rank_error'].median()

print(f'Mean absolute rank error:  {mean_abs_rank_err:.2f} places')
print(f'Median absolute rank error:{median_rank_err:.2f} places')
print(f'Mean absolute score error: {mean_abs_score_err:.2f} points (normalized)')
print()

# ── Spearman rank correlation per event ───────────────────────
print('=== Spearman Rank Correlation by Event/Games ===')
print()
print(f'{"Event + Games":<45} {"N":<6} {"Spearman rho":<16} Interpretation')
print('-' * 85)

spearman_results = []
for (event, games), grp in full_event_results.groupby(['event', 'games']):
    if len(grp) < 3:
        continue
    rho, pval = spearmanr(grp['pred_rank'], grp['actual_rank'])
    label = f'{event} — {games}'
    interp = ('strong' if abs(rho) > 0.7
               else 'moderate' if abs(rho) > 0.4
               else 'weak')
    direction = 'positive' if rho > 0 else 'NEGATIVE'
    print(f'{label:<45} {len(grp):<6} {rho:<16.3f} {interp} {direction} (p={pval:.3f})')
    spearman_results.append({'event': event, 'games': games,
                              'n': len(grp), 'rho': rho, 'pval': pval})

spearman_df = pd.DataFrame(spearman_results)
print()
if not spearman_df.empty:
    print(f'Mean Spearman rho across events: {spearman_df["rho"].mean():.3f}')
    print()
print('Interpretation: rho=1.0 means perfect rank ordering, rho=-1.0 means completely inverted')
print('The model can be well-calibrated in score but still get ranks wrong if the field is tight')
print()

# ── Preparation gap archetype deep dive ─────────────────────────────────────────
malinin_row = full_event_results[
    (full_event_results['name'] == 'DOMINANT_FIRST') &
    (full_event_results['games'] == 'Games_2026')
]
if not malinin_row.empty:
    m = malinin_row.iloc[0]
    print('=== DOMINANT_FIRST 2026 — Rank Error Deep Dive ===')
    print(f'  Predicted score:  {m.pred_score:.1f}')
    print(f'  Predicted rank:   {m.pred_rank} (highest in field)')
    print(f'  Actual rank:      {m.actual_rank}')
    print(f'  Rank error:       {m.rank_error:+.0f} places')
    print(f'  Score error:      {m.score_error:+.1f} points (normalized)')
    print()
    print('  The rank error of +7 is more informative than the score error alone.')
    print('  He did not just face elevated difficulty his projection — he face elevated difficultyed')
    print('  relative to the entire field simultaneously.')
    print('  This is the signature of a pressure performance shift, not random variance.')
    print()
    print('  A model that predicted rank error > 3 before the competition')
    print('  would have justified the hedge recommendation in the optimization.')


In [ ]:
# ── Rank error visualization ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Rank Error Analysis — Predicted vs Actual Placement',
             fontsize=13, fontweight='bold')

# ── Plot 1: Predicted rank vs actual rank scatter
ax = axes[0]
perfect = [0, full_event_results['actual_rank'].max() + 1]
ax.plot(perfect, perfect, 'k--', alpha=0.3, label='Perfect prediction')

for _, row in full_event_results.iterrows():
    is_malinin = row['name'] == 'DOMINANT_FIRST' and row['games'] == 'Games_2026'
    color = 'crimson' if is_malinin else 'steelblue'
    size  = 150 if is_malinin else 70
    ax.scatter(row['actual_rank'], row['pred_rank'], color=color, s=size, zorder=5)
    ax.annotate(f"{row['name'].split()[-1]} {row['games'][-4:]}",
                (row['actual_rank'], row['pred_rank']),
                textcoords='offset points', xytext=(4, 2), fontsize=7)

ax.set_xlabel('Actual Rank')
ax.set_ylabel('Predicted Rank')
ax.set_title('Predicted vs Actual Rank\n(below line = predicted too high)')
ax.invert_xaxis(); ax.invert_yaxis()  # rank 1 at top-left

# ── Plot 2: Rank error bar chart
ax = axes[1]
sorted_df = full_event_results.sort_values('rank_error')
labels = [f"{r['name'].split()[-1]} {r['games'][-4:]}" for _, r in sorted_df.iterrows()]
bar_colors = ['crimson' if e > 3 else '#5cb85c' if e < -1 else 'steelblue'
               for e in sorted_df['rank_error']]
ax.barh(labels, sorted_df['rank_error'], color=bar_colors, alpha=0.85)
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Rank Error (predicted − actual)')
ax.set_title('Rank Error by Athlete\n(+= predicted too high, −= predicted too low)')

# ── Plot 3: Score error vs rank error
ax = axes[2]
for _, row in full_event_results.iterrows():
    is_malinin = row['name'] == 'DOMINANT_FIRST' and row['games'] == 'Games_2026'
    color = 'crimson' if is_malinin else 'steelblue'
    size  = 150 if is_malinin else 70
    ax.scatter(row['abs_score_error'], row['abs_rank_error'],
               color=color, s=size, zorder=5)
    ax.annotate(f"{row['name'].split()[-1]}",
                (row['abs_score_error'], row['abs_rank_error']),
                textcoords='offset points', xytext=(4, 2), fontsize=7)

ax.set_xlabel('Absolute Score Error (normalized points)')
ax.set_ylabel('Absolute Rank Error (places)')
ax.set_title('Score Error vs Rank Error\n(DOMINANT_FIRST in red — both high)')
ax.text(0.05, 0.95, 'Top right = worst misses', transform=ax.transAxes,
        fontsize=9, verticalalignment='top', color='gray')

plt.tight_layout()
plt.savefig('data/outputs/rank_error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Rank error visualization saved')
print()
print('=== Combined Error Summary ===')
print(f'Mean absolute score error: {full_event_results["abs_score_error"].mean():.2f} normalized pts')
print(f'Mean absolute rank error:  {full_event_results["abs_rank_error"].mean():.2f} places')
if not spearman_df.empty:
    print(f'Mean Spearman rho:         {spearman_df["rho"].mean():.3f}')
print()
print('Key insight: rank error catches what score error misses.')
print('A 40-point score miss on DOMINANT_FIRST is a miss.')
print('A rank-7 error on DOMINANT_FIRST tells you the whole field outperformed him —')
print('which is the signature of pressure performance shift, not random bad day.')


## Section 7 — Master Dashboard

In [ ]:
# ── Master dashboard ──────────────────────────────────────────
fig = plt.figure(figsize=(20, 16))
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.38)
fig.suptitle('USOPC Analytics Pipeline — LA 2028 Summer Games Dashboard',
             fontsize=15, fontweight='bold')

colors = {'gold': '#FFD700', 'usa': '#003087', 'good': '#5cb85c',
          'bad': '#d9534f', 'neutral': '#5bc0de'}

# ── 1: Prediction vs actual
ax1 = fig.add_subplot(gs[0, :2])
if 'pred_score_nb2' in val_df.columns and len(val_df) > 0:
    perfect = [0, 110]
    ax1.plot(perfect, perfect, 'k--', alpha=0.3, label='Perfect')
    gold_mask = val_df['actual_gold'] == 1 if 'actual_gold' in val_df.columns else [False]*len(val_df)
    for is_gold, grp in val_df.groupby(gold_mask):
        col = colors['gold'] if is_gold else colors['usa']
        ax1.scatter(grp['actual_score'], grp['naive_pred'],
                    c=col, marker='o', s=70, alpha=0.7, label=f'Naive {"Gold" if is_gold else ""}')
        ax1.scatter(grp['actual_score'], grp['pred_score_nb2'],
                    c=col, marker='^', s=70, alpha=0.7, label=f'NB2 {"Gold" if is_gold else ""}')
    for _, r in val_df.iterrows():
        ax1.annotate(f"{r['name'].split()[-1]} {r['games'][-4:]}",
                     (r['actual_score'], r['pred_score_nb2']),
                     textcoords='offset points', xytext=(4,2), fontsize=6)
    ax1.set_xlabel('Actual Score')
    ax1.set_ylabel('Predicted Score')
    ax1.set_title('Prediction Accuracy: Naive (●) vs Model (▲)')
else:
    ax1.text(0.5, 0.5, 'Run validation with real data', ha='center', va='center',
             transform=ax1.transAxes, fontsize=12)
    ax1.set_title('Prediction vs Actual (needs data)')

# ── 2: Error comparison bar
ax2 = fig.add_subplot(gs[0, 2])
if naive_mae > 0:
    ax2.bar(['Naive MAE', 'NB2 MAE'], [naive_mae, nb2_mae],
            color=[colors['bad'], colors['good']], alpha=0.85)
    ax2.set_ylabel('Score Points')
    ax2.set_title('Prediction Error: Naive vs Model')
    for i, v in enumerate([naive_mae, nb2_mae]):
        ax2.text(i, v + 0.3, f'{v:.1f}', ha='center', fontweight='bold')
else:
    ax2.text(0.5, 0.5, 'Needs validation data', ha='center', va='center',
             transform=ax2.transAxes)
    ax2.set_title('Error Comparison')

# ── 3: Efficient frontier
ax3 = fig.add_subplot(gs[1, :2])
if not frontier_df.empty:
    ax3.plot(frontier_df['budget'], frontier_df['exp_golds'],
             color=colors['usa'], linewidth=2.5, marker='o', markersize=4)
    if BUDGET <= frontier_df['budget'].max():
        ax3.axvline(BUDGET, color='red', linestyle='--', alpha=0.7, label=f'Budget={BUDGET}')
    ax3.fill_between(frontier_df['budget'], 0, frontier_df['exp_golds'],
                     alpha=0.1, color=colors['usa'])
    ax3.set_xlabel('Budget (investment units)')
    ax3.set_ylabel('Expected Golds')
    ax3.set_title('Efficient Frontier — Games_2026 (LP + greedy rounding, 40 steps)')
    ax3.legend()
else:
    ax3.text(0.5, 0.5, 'Run optimization with real data', ha='center', va='center',
             transform=ax3.transAxes, fontsize=11)
    ax3.set_title('Efficient Frontier (needs data)')

# ── 4: Policy simulation bar
ax4 = fig.add_subplot(gs[1, 2])
if not scenario_df.empty:
    scenario_colors = [colors['bad'] if 'Baseline' in s
                       else colors['neutral'] if 'Protocol' not in s
                       else colors['good']
                       for s in scenario_df['scenario']]
    bars = ax4.barh(scenario_df['scenario'], scenario_df['p_gold'],
                    color=scenario_colors, alpha=0.85)
    ax4.set_xlabel('P(Gold) %')
    ax4.set_title('DOMINANT_FIRST 2026 — Investment Scenarios')
    ax4.axvline(scenario_df['p_gold'].iloc[0], color='red',
                linestyle='--', alpha=0.5)
    for bar, val in zip(bars, scenario_df['p_gold']):
        ax4.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=8)
else:
    ax4.text(0.5, 0.5, 'Run policy simulation', ha='center', va='center',
             transform=ax4.transAxes)
    ax4.set_title('Policy Simulation (needs data)')

# ── 5: P(gold) vs P(medal) scatter
ax5 = fig.add_subplot(gs[2, :2])
if not athletes_df.empty and 'p_gold' in athletes_df.columns:
    selected_mask = athletes_df.get('selected', pd.Series([0]*len(athletes_df))) == 1
    for is_sel, grp in athletes_df.groupby(selected_mask):
        col  = colors['usa']  if is_sel else colors['neutral']
        mark = 'D' if is_sel else 'o'
        size = 120 if is_sel else 70
        ax5.scatter(grp['p_gold'], grp['p_medal'],
                    c=col, marker=mark, s=size, alpha=0.85,
                    label='Selected' if is_sel else 'Not selected', zorder=5)
    for _, r in athletes_df.iterrows():
        ax5.annotate(r['name'].split()[-1],
                     (r['p_gold'], r['p_medal']),
                     textcoords='offset points', xytext=(4, 2), fontsize=7)
    ax5.set_xlabel('P(Gold) %')
    ax5.set_ylabel('P(Medal) %')
    ax5.set_title('Athlete Portfolio: P(Gold) vs P(Medal) (◆=selected, ●=not selected)')
    ax5.legend()
else:
    ax5.text(0.5, 0.5, 'Populate athlete data', ha='center', va='center',
             transform=ax5.transAxes)
    ax5.set_title('Portfolio Map (needs data)')

# ── 6: Scorecard
ax6 = fig.add_subplot(gs[2, 2])
ax6.axis('off')
scorecard = [
    ['Metric',           'Value'],
    ['Athletes modeled', str(len(athletes_df)) if not athletes_df.empty else '—'],
    ['Validation N',     str(len(val_df)) if len(val_df) > 0 else '—'],
    ['Naive MAE (pts)',  f'{naive_mae:.1f}' if naive_mae > 0 else '—'],
    ['NB2 MAE (pts)',    f'{nb2_mae:.1f}'   if nb2_mae   > 0 else '—'],
    ['Brier Skill',      f'{nb1_bss:.3f}'   if nb1_bss   != 0 else '—'],
    ['Frontier steps',   str(len(frontier_df)) if not frontier_df.empty else '—'],
    ['Budget',           str(BUDGET)],
    ['Data sources',     'ISU/FIS/WA/Olympedia'],
]
tbl = ax6.table(cellText=scorecard[1:], colLabels=scorecard[0],
                loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.2, 1.9)
ax6.set_title('Pipeline Scorecard', fontweight='bold', pad=20)

output_path = f'{OUTPUT_DIR}/usopc_master_dashboard.png'
plt.savefig(output_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Dashboard saved to {output_path}')


In [ ]:
# ── Save master results CSV ───────────────────────────────────
if not athletes_df.empty:
    save_cols = [c for c in [
        'athlete_id', 'name', 'nation', 'discipline', 'games', 'season',
        'mean_pre', 'std_pre', 'best_pre', 'comps_pre',
        'age', 'age_vs_peak', 'first_olympics', 'prior_olympics',
        'win_streak', 'sentiment',
        'pred_mean', 'pred_std', 'nlp_adj',
        'p_gold', 'p_medal', 'p_top5',
        'cost', 'qual_slot', 'selected',
    ] if c in athletes_df.columns]

    out_path = f'{OUTPUT_DIR}/usopc_master_results.csv'
    athletes_df[save_cols].to_csv(out_path, index=False)
    print(f'✅ Master results saved to {out_path}')

if not frontier_df.empty:
    frontier_df.to_csv(f'{OUTPUT_DIR}/usopc_frontier.csv', index=False)
    print(f'✅ Frontier saved to {OUTPUT_DIR}/usopc_frontier.csv')

print()
print('=' * 65)
print('USOPC ANALYTICS PIPELINE — COMPLETE')
print('=' * 65)
print()
print('Section 0: Data ingestion (ISU, FIS, World Athletics, Olympedia)')
print('Section 1: Feature engineering (age curves, consistency, NLP)')
print('Section 2: Performance prediction (GBM + quantile + NLP sentiment)')
print('Section 3: Monte Carlo simulation (P(gold), P(medal), P(top5))')
print('Section 4: LP/IP optimization (frontier + portfolio selection)')
print('Section 5: Policy simulation (investment → performance response)')
print('Section 6: Validation (Brier, MAE, skill score vs baseline)')
print('Section 7: Master dashboard')
print()
print('To improve the model:')
print('  1. Replace PLACEHOLDER data in Section 0 with real scraped scores')
print('  2. Add actual_results rows in Section 6 after each Games')
print('  3. Add pre-Games statements in Section 2 NLP block')
print('  4. Tune competitor_fields in Section 3 with real international data')
